# HD Integrated Visium Interactive Suite (HiVis) - Annotations and classifiers
This Notebook describes how to import single-cell segmentation from Qupath into HiVis. Segmentation can be both Stardist and Cellpose.

We will use the HiVis object from [previous notebook]().

We will also use some plots, which will be covered in detail in next notebooks.

To see a details explanation and parameters for each function, visit the [documentation](https://hivis.readthedocs.io/en/latest/).

In [ ]:
import os
import sys
import matplotlib.pyplot as plt
from HiVis import HiVis

sys.path.append(os.path.abspath(".."))

path = 'output/mouse_intestine.pkl'
si = HiVis.load(path) 

We can aggregate the spots into single-cells or single-nuclei.

In this example, the segmentation is performed in Qupath via Stardist with expansion of nuclei.

Tutorial for the segmentation can be seen in the [readme file]().

In [ ]:
segmentation_path = "qupath/stardist_results.csv"
segmentation = pd.read_csv(segmentation_path, sep="\t")
segmentation.rename(columns={"InCell":"in_cell", "InNuc":'in_nucleus',"Object ID":"Cell_ID"}, inplace=True)

The aggregation by single cells usually takes few minutes.

Notice that we can add measurements from qupath (color intensity, diameter of cells etc.) to to HiVis adata, and also aggregate metadata from the HiVis.
In this example, we will transfer the annotations and the classifier.

We can also specify the path of the GeoJSON, to have the geometry of the single-cells (usefull for plotting). We can also skip this, and add the geometry later, by calling HiVis.import_geometry(geojson_path).

In [ ]:
geojson_path = r"qupath/geometry.geojson"
si.agg_stardist(segmentation, name="SC", obs2add=["Area µm^2","Length µm"], obs2agg=["intestine_part","muscle_villi_classifier"],geojson_path=geojson_path)

In [ ]:
We can acess the new Aggregation 

In [ ]:
si_subset.agg["SC"]

In [ ]:
print(si.name)
print(si.shape) # spots * genes
print(si.path_output)

The Aggregation is linked to the HiVis object

In [ ]:
si_subset.agg["SC"].viz

We can plot the cells (detailed tutorial is in next notebooks)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = {"immune":"blue", "lumen":"gray", "muscle":"red", "tissue":"orange"}
si_subset.agg["SC"].plot.spatial("muscle_villi_classifier", xlim=[3000,3600], ylim=[300,900], ax=axes[0], cmap=colors, legend=False, alpha=0.7, axis_labels=False,size=20)
si_subset.agg["SC"].plot.hist("muscle_villi_classifier", ax=axes[1], cmap=colors, ylab="Cells count")
si_subset.agg["SC"].plot.cells("muscle_villi_classifier", xlim=[3000,3600], ylim=[300,900], ax=axes[2],alpha=0.7, axis_labels=False, cmap=colors)
plt.tight_layout()

## Single-cell clustering with other tools

## Transfer information from cells to bins

## Subsetting, copying and export
Similar to subsetting of HiVis, we can subset and copy Aggregations

In [ ]:
cluster_0 = si_subset.agg["SC"]["leiden"] == "0"
si_subset.agg["cluster_0"] = si_subset.agg["SC"][cluster_0,:]
si_subset.agg["cluster_0"].rename("mouse_intestine_SC_clust0", full=True, new_out_path=True)

In [ ]:
si_subset.agg["cluster_0_copy"] = si_subset.agg["cluster_0"].copy(name="copy")

In [ ]:
si.save()